# Vyuha P15 - NIST-AI-RMF safety-guard benchmark

Reproduces the protocol of **Harsh, Sarmah & Pasquali, *Benchmarking Open-Source Safety Guard Models*** (arXiv:2605.28830, ICLR 2026 workshop): score a content guard on the **8 NIST AI RMF safety categories** and report **recall** - the paper's headline metric (a missed harmful item costs more than a false positive). The paper's best is **Qwen Guard 4B at 83.97% recall**; larger guards are *more* conservative (miss up to 75%), so **model size does not predict recall**, and it **recommends ensembling non-overlapping guards** - which Vyuha's L2 ensemble (P13) already does.

**Honesty note.** The paper's exact 79,331-sample filtered split is **not released**, so this notebook runs an *approximate reconstruction* from the public sources it names (here, BeaverTails as a categorized source - a subset of the paper's four). Numbers are Vyuha's guard on this reconstruction, **not** the official split; treat them as indicative and label them as such. Needs a **GPU** (Kaggle T4) for the guard.

In [ ]:
import sys, os, glob, subprocess
REPO_URL = "https://github.com/g25ait2149/vyuha.git"
DEST = "/kaggle/working/vyuha_src"
if os.path.isdir(os.path.join(DEST, ".git")):
    subprocess.run(["git", "-C", DEST, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, DEST], check=False)
hits = glob.glob(DEST + "/**/vyuha/__init__.py", recursive=True)
root = os.path.dirname(os.path.dirname(hits[0])) if hits else DEST
sys.path.insert(0, root)
for m in [m for m in sys.modules if m == "vyuha" or m.startswith(("vyuha.", "eval"))]:
    del sys.modules[m]
print("vyuha repo at:", root)

## A. Offline check (no GPU) - the recall aggregation is correct
A stub guard blind to one category -> that category's recall is 0, the rest 1.0, overall 0.875. Verifies the harness before spending GPU.

In [ ]:
from eval.nist_rmf_eval import _demo
_demo(verbose=True)

## B. Build a categorized subset (BeaverTails -> NIST-RMF)
BeaverTails carries per-sample harm-category labels; we map the ones that align to the NIST-RMF axes. **Verify the category keys against the loaded schema** (printed below) and adjust `BT_TO_NIST` - dataset fields can change. Categories with no BeaverTails match (e.g. profanity, harassment) need the paper's other sources (RealToxicityPrompts) for full 8-axis coverage; that is left as an extension so nothing here is fabricated.

In [ ]:
from datasets import load_dataset
ds = load_dataset('PKU-Alignment/BeaverTails', split='30k_test')   # public; adjust split if needed
print('features:', ds.features)
print('example category field:', ds[0].get('category'))

In [ ]:
# best-effort, EDITABLE mapping from BeaverTails harm categories -> NIST-RMF axes (verify above)
BT_TO_NIST = {
    'violence_aiding_and_abetting_incitement': 'violence',
    'hate_speech_offensive_language': 'hate_speech',
    'discrimination_stereotype_injustice': 'hate_speech',
    'sexually_explicit_adult_content': 'sexual_content',
    'self_harm': 'self_harm',
    'misinformation_regarding_ethics_laws_and_safety': 'health_misinformation',
    'terrorism_organized_crime': 'threats',
}
N_UNSAFE, N_BENIGN = 800, 800     # cap for a T4-feasible run; raise for more coverage
texts, labels, cats = [], [], []
for row in ds:
    cat_dict = row.get('category') or {}
    active = [BT_TO_NIST[k] for k, v in cat_dict.items() if v and k in BT_TO_NIST]
    is_safe = row.get('is_safe', None)
    if active and is_safe is False and sum(labels) < N_UNSAFE:
        texts.append(row['prompt']); labels.append(1); cats.append(active[0])
    elif is_safe is True and (len(labels) - sum(labels)) < N_BENIGN:
        texts.append(row['prompt']); labels.append(0); cats.append('benign')
print(f'unsafe={sum(labels)}  benign={len(labels)-sum(labels)}  categories={sorted(set(c for c in cats if c!="benign"))}')

## C. Score Vyuha's L2 content guard (Qwen3Guard) - recall per NIST-RMF category

In [ ]:
from vyuha.guard import OpenGuard
from eval.nist_rmf_eval import nist_rmf_benchmark
guard = OpenGuard.preset('qwen3guard')      # Vyuha's L2 content guard (0.6B)
rep = nist_rmf_benchmark(guard, texts, labels, cats, verbose=True)
rep['overall_recall'], rep['macro_recall'], rep['benign_fpr']

## D. Test the paper's recommendation: ensemble two non-overlapping guards (Vyuha P13)
The paper recommends ensembling non-overlapping guards. Vyuha's `GuardEnsemble` unions Qwen3Guard with a second guard; recall should rise (the union catches what either misses) - the P13 complementarity claim, measured on the NIST-RMF axes.

In [ ]:
from vyuha.guard import GuardEnsemble
members = {g: OpenGuard.preset(g) for g in ['qwen3guard', 'granite-guardian']}
ens = GuardEnsemble(members, mode='max')
rep_ens = nist_rmf_benchmark(ens, texts, labels, cats, verbose=True)
print('single-guard recall:', rep['overall_recall'], '-> ensemble recall:', rep_ens['overall_recall'])

## Interpretation
- Report **recall** as the headline (per the paper), with **benign FPR** alongside so a high-recall guard isn't just blocking everything.
- Vyuha's L2 is **Qwen3Guard-0.6B**, smaller than the paper's 4B leader (83.97%); expect lower recall - state the size honestly.
- If the **ensemble** raises recall over the single guard, that is the paper's *ensemble non-overlapping guards* recommendation, measured - direct external validation of Vyuha's P13 design.
- These numbers are on a **BeaverTails-derived reconstruction**, not the official 79,331-sample split; label them as indicative.